In [ ]:
import os
import csv
import hashlib
import requests
import imagehash

from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime
from io import BytesIO
from PIL import Image, ImageOps
from urllib3.exceptions import InsecureRequestWarning


from IPython.display import clear_output, display as display_image

from googleapiclient.discovery import build

import notebook_setup  # isort:skip
from acid import conf  # isort:skip
from acid.torch.setup import BoardModelSetup  # isort:skip

In [ ]:
GOOGLE_API_KEY = os.environ["AC_GOOGLE_API_KEY"]
GOOGLE_SEARCH_ENGINE_ID = os.environ["AC_GOOGLE_SEARCH_ENGINE_ID"]
IMAGES_SAVE_DIR = conf.TRAINING_DATA_DIR / "boards" / "from_google"
ATTRIBUTION_CSV_PATH = conf.TRAINING_DATA_DIR / "boards" / "attribution.csv"
DL_STATUS_CSV_PATH = conf.TRAINING_DATA_DIR / "boards" / "google_dl_status.csv"
IMAGE_SIZE = BoardModelSetup.image_size

SEARCH_QUERY = "chess board"
PAGE_START = 1
PAGE_END = 10

# we are running requests with verify=False => suppress warnings
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

In [ ]:
# https://developers.google.com/custom-search/v1/reference/rest/v1/cse/list
search_params = {
    "q": SEARCH_QUERY,
    "cx": GOOGLE_SEARCH_ENGINE_ID,
    "searchType": "image",
    "num": 10,
    "imgType": "photo",
    "safe": "off",
    "rights": "cc_publicdomain|cc_attribute|cc_sharealike|cc_noncommercial|cc_nonderived",
}

status_csv_fields = [
    "source_url",
    "source_referrer_url",
    "source_name",
    "source_hash_sha1",
    "source_image_hash_whash",
    "path_rel",
    "skipped",
    "processed_at",
]
google_api = build("customsearch", "v1", developerKey=GOOGLE_API_KEY)
image_hashes = set()

# read existing image_hashes
if DL_STATUS_CSV_PATH.exists():
    with open(DL_STATUS_CSV_PATH, "r", newline="") as status_fh:
        status_reader = csv.DictReader(status_fh)
        for row in status_reader:
            image_hashes.add(imagehash.hex_to_hash(row["source_image_hash_whash"]))

# get results from api
with open(DL_STATUS_CSV_PATH, "a", newline="") as status_fh:
    status_writer = csv.DictWriter(status_fh, fieldnames=status_csv_fields)
    if not status_fh.tell():
        status_writer.writeheader()

    page = PAGE_START
    while True:
        search_params["start"] = page
        api_response = google_api.cse().list(**search_params).execute()

        for result in api_response.get("items", []):
            clear_output()

            image_referrer_url = result["image"]["contextLink"]
            image_url = result["link"]

            print(f"Processing page {page} - {image_url}")
            try:
                response = requests.get(image_url, verify=False)
            except (TimeoutError, ReadTimeout):
                continue
            if response.status_code != 200:
                continue

            image_raw = response.content
            image_hash_sha1 = hashlib.sha1(image_raw).hexdigest()
            image_pil = Image.open(BytesIO(image_raw))
            image_hashes_whash = imagehash.whash(image_pil)

            image_path = IMAGES_SAVE_DIR / image_hash_sha1[0:2] / f"{image_hash_sha1}.jpeg"
            if image_path.exists():
                continue

            # this is not performant but works for the purpose ;)
            skip_image = False
            for image_hash in image_hashes:
                if image_hash - image_hashes_whash <= 3:
                    print(f"Skipping image {image_url}")
                    skip_image = True
                    break

            if skip_image:
                continue

            image_hashes.add(image_hashes_whash)
            image_pil = ImageOps.pad(image_pil, IMAGE_SIZE, color="white")
            display_image(image_pil)
            save_image = input("save this image? (y): ")
            if save_image == "n":
                save_image = False
            else:
                save_image = True

            if save_image:
                image_path.parent.mkdir(parents=True, exist_ok=True)
                image_pil = image_pil.convert("RGB")
                image_pil.save(image_path.resolve(), format="jpeg")
            else:
                image_path = None

            status_writer.writerow(
                {
                    "source_url": image_url,
                    "source_referrer_url": image_referrer_url,
                    "source_name": Path(urlparse(image_url).path).name,
                    "source_hash_sha1": str(image_hash_sha1),
                    "source_image_hash_whash": str(image_hashes_whash),
                    "path_rel": str(image_path),
                    "skipped": int(not save_image),
                    "processed_at": datetime.now(),
                }
            )

            status_fh.flush()

        page += 1
        if page >= PAGE_END:
            break

In [ ]:
# write final attributions csv
with open(DL_STATUS_CSV_PATH, "r", newline="") as in_fh, open(ATTRIBUTION_CSV_PATH, "w", newline="") as out_fh:
    status_reader = csv.DictReader(in_fh)
    attr_writer = csv.DictWriter(out_fh, fieldnames=status_csv_fields)
    attr_writer.writeheader()
    for row in status_reader:
        if not int(row["skipped"]):
            attr_writer.writerow(row)

print("done!")